# **Lab 6. Camera**


## **The objectives of today's lab**

* Learn how to connect to the camera and retrieve images via code
* Learn how to calibrate the camera
* Learn how to search for objects in images

## **What you need to do**

1. Statically position the camera so that it is pointing parallel to the table. Set up uniform lighting.
2. Shoot a minimum of 40 frames of the chessboard in different positions relative to the camera. You can print chessboard according to this [link](https://oir.mobi/uploads/posts/2021-03/1616590390_14-p-shakhmatnaya-doska-fon-15.png)
3. Obtain a minimum of 10 different images of the coins on the table in different positions
4. calibrate the camera
5. Write a coin detection algorithm and detect all coins
6. Write an algorithm to recognize coins and evaluate the accuracy of the algorithm by the metric: the total value of coins in the image recognized by the algorithm in relation to the real value.

## **Bonus Task [5 points]**

1. Create a new dataset, with a non-static camera position. It should have one object of a known size to determine the distance between objects and the camera, as well as coins for recognition.
2. Supplement the algorithm where you should calculate the real distance between the camera and objects. Calculate the real size of the coins, and update your algorithm for recognition.
3. A dataset of 10 images, an object of your choice.

## **Lab hardware**

Camera Full HD 4k

## **Lab software**

We will use [OpenCV](https://opencv.org/get-started/) to get data from camera and process it

## **Userful links**
https://developer.ridgerun.com/wiki/index.php?title=How_to_Capture_Frames_from_Camera_with_OpenCV_in_Python

https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html

https://www.geeksforgeeks.org/circle-detection-using-opencv-python/

## **Part 1: Camera Calibration with Your Dataset**
Using your existing stereo camera dataset from the homework.

In [ ]:
import cv2
import numpy as np
import glob
import os
from pathlib import Path
import matplotlib.pyplot as plt

# Chessboard definition from your previous work
CHECKERBOARD = (8, 5)  # Inner corners (columns, rows) = (8, 5)
SQUARE_SIZE_MM = 25.0  # mm

# Dataset path
DATASET_PATH = "Dataset/checkerBoardDataset"

# Ensure we use absolute path if needed
if not os.path.exists(DATASET_PATH):
    # Try alternate path relative to current working directory
    DATASET_PATH = "/Users/anashamrouni/Documents/InnopolisUniversity/Bachelor/SensorsAndSensing/LAB07_CAMERA/Dataset/checkerBoardDataset"

print(f"Dataset path: {DATASET_PATH}")
print(f"Dataset exists: {os.path.exists(DATASET_PATH)}")
print(f"Images in dataset: {len(glob.glob(os.path.join(DATASET_PATH, '*.jpg')))}")


In [ ]:
def calibrate_camera(image_paths, checkerboard, square_size_mm):
    """
    Calibrate camera using chessboard images.
    
    Parameters:
    - image_paths: List of paths to calibration images
    - checkerboard: Tuple (cols, rows) of inner corners
    - square_size_mm: Size of chessboard square in mm
    
    Returns:
    - camera_matrix (K): Intrinsic parameters
    - distortion_coeff (dist): Distortion coefficients
    - mean_error: Mean reprojection error in pixels
    - rvecs, tvecs: Rotation and translation vectors
    """
    # Prepare 3D points using known square size
    objp = np.zeros((checkerboard[0] * checkerboard[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:checkerboard[0], 0:checkerboard[1]].T.reshape(-1, 2)
    objp *= square_size_mm  # Convert to mm
    
    objpoints = []  # 3D points in world space
    imgpoints = []   # 2D points in image space
    
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
    
    valid_count = 0
    for img_path in image_paths:
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read {img_path}")
            continue
            
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Find chessboard corners
        ret, corners = cv2.findChessboardCorners(gray, checkerboard, None)
        
        if ret:
            # Refine corner positions to sub-pixel precision
            corners_refined = cv2.cornerSubPix(
                gray, corners, (11, 11), (-1, -1), criteria
            )
            objpoints.append(objp)
            imgpoints.append(corners_refined)
            valid_count += 1
        else:
            print(f"Corners NOT found in: {os.path.basename(img_path)}")
    
    print(f"\nValid calibration images: {valid_count}/{len(image_paths)}")
    
    if valid_count < 5:
        raise RuntimeError(f"Too few valid images ({valid_count}). Need at least 5.")
    
    # Calibrate camera
    h, w = gray.shape[:2]
    ret, camera_matrix, distortion_coeff, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, (w, h), None, None
    )
    
    # Calculate mean reprojection error
    total_error = 0.0
    for i in range(len(objpoints)):
        proj, _ = cv2.projectPoints(
            objpoints[i], rvecs[i], tvecs[i], camera_matrix, distortion_coeff
        )
        error = cv2.norm(imgpoints[i], proj, cv2.NORM_L2) / len(proj)
        total_error += error
    
    mean_error = total_error / len(objpoints)
    
    return camera_matrix, distortion_coeff, mean_error, rvecs, tvecs

print("Calibration function defined.")


In [ ]:
# Load left camera images (using L*.jpg from stereo dataset)
left_images = sorted(glob.glob(os.path.join(DATASET_PATH, "L*.jpg")))

print(f"Found {len(left_images)} left camera images")
if left_images:
    print(f"First image: {os.path.basename(left_images[0])}")
    print(f"Last image: {os.path.basename(left_images[-1])}")


In [ ]:
# Run calibration
print("Starting calibration with chessboard (8, 5) and square size 25mm...")
K, dist, mean_error, rvecs, tvecs = calibrate_camera(
    left_images, CHECKERBOARD, SQUARE_SIZE_MM
)

print("\n" + "="*60)
print("CALIBRATION RESULTS")
print("="*60)
print(f"\nMean Reprojection Error: {mean_error:.4f} pixels")
if mean_error < 0.3:
    print("✓ Excellent calibration quality (< 0.3 px)")
elif mean_error < 0.5:
    print("✓ Good calibration quality (< 0.5 px)")
elif mean_error < 1.0:
    print("⚠ Acceptable calibration quality (< 1.0 px)")
else:
    print("✗ Poor calibration quality (> 1.0 px) - consider recapturing images")

print(f"\nCamera Matrix (K):\n{K}")
print(f"\nDistortion Coefficients:\n{dist.ravel()}")


In [ ]:
# Save calibration results
output_dir = "calibration_results"
os.makedirs(output_dir, exist_ok=True)

calib_file = os.path.join(output_dir, "camera_calibration.npz")
np.savez(calib_file, K=K, dist=dist, mean_error=mean_error)
print(f"\nCalibration saved to: {calib_file}")

# Also save as readable text
txt_file = os.path.join(output_dir, "calibration_params.txt")
with open(txt_file, "w") as f:
    f.write("CAMERA CALIBRATION PARAMETERS\n")
    f.write("=" * 50 + "\n")
    f.write(f"Chessboard: {CHECKERBOARD} (inner corners)\n")
    f.write(f"Square Size: {SQUARE_SIZE_MM} mm\n")
    f.write(f"Images used: {len(left_images)}\n")
    f.write(f"Mean Reprojection Error: {mean_error:.4f} px\n\n")
    f.write("Camera Matrix (K):\n")
    f.write(str(K) + "\n\n")
    f.write("Distortion Coefficients:\n")
    f.write(str(dist.ravel()) + "\n")

print(f"Parameters also saved to: {txt_file}")


In [ ]:
# Verify undistortion on a sample image
test_img_path = left_images[0]
img = cv2.imread(test_img_path)

h, w = img.shape[:2]

# Compute undistortion map (for efficient batch processing)
mapx, mapy = cv2.initUndistortRectifyMap(K, dist, None, K, (w, h), cv2.CV_32F)

# Undistort the image
img_undistorted = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)

# Display comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original (Distorted)")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(img_undistorted, cv2.COLOR_BGR2RGB))
axes[1].set_title("Undistorted")
axes[1].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "undistortion_example.png"), dpi=100, bbox_inches="tight")
plt.show()

print("Undistortion verification complete. Compare the straight edges!")


## **Part 2: Coin Detection & Recognition**

**Key workflow:**
1. Load coin images (from `Dataset/coinImages` or your own captures)
2. Undistort each image using K and dist
3. Detect circles (coins) using Hough Circle Detection
4. Classify coins by radius/diameter → value
5. Calculate total value and compare to ground truth

In [ ]:
def detect_coins_hough(img_undistorted, min_radius=20, max_radius=80):
    """
    Detect coins using Hough Circle Detection.
    
    Parameters:
    - img_undistorted: Undistorted image (BGR)
    - min_radius, max_radius: Expected coin radius range in pixels
    
    Returns:
    - circles: Array of detected circles (x, y, radius)
    """
    gray = cv2.cvtColor(img_undistorted, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    
    # Hough Circle Detection
    circles = cv2.HoughCircles(
        blurred,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=min_radius,
        param1=100,  # Upper threshold for Canny edge detection
        param2=30,   # Accumulator threshold
        minRadius=min_radius,
        maxRadius=max_radius
    )
    
    if circles is not None:
        circles = np.uint16(np.around(circles))
        return circles[0]
    else:
        return np.array([])

def classify_coins_by_radius(circles, radius_to_value):
    """
    Classify detected coins by radius.
    
    Parameters:
    - circles: Array of circles from detection
    - radius_to_value: Dict mapping radius ranges to coin values
      Example: {(15, 25): 5, (25, 35): 10, (35, 45): 25}
    
    Returns:
    - classified: List of (x, y, radius, value) tuples
    - total_value: Sum of detected coin values
    """
    classified = []
    total_value = 0
    
    for circle in circles:
        x, y, r = circle
        
        for (r_min, r_max), value in radius_to_value.items():
            if r_min <= r <= r_max:
                classified.append((x, y, r, value))
                total_value += value
                break
    
    return classified, total_value

print("Coin detection and classification functions defined.")


In [ ]:
# Example: Define coin radius-to-value mapping
# ADJUST THESE VALUES based on your coin measurements from undistorted images
# Measure coin diameters in pixels and match them to real coin values

RADIUS_TO_VALUE = {
    # (min_radius_px, max_radius_px): coin_value_in_currency
    (15, 22): 1,      # 1 ruble
    (22, 28): 5,      # 5 rubles
    (28, 35): 10,     # 10 rubles
    (35, 45): 50,     # 50 rubles (adjust based on your measurements)
}

print("Coin classification mapping defined.")
print("IMPORTANT: Measure coin radii in your undistorted images and update RADIUS_TO_VALUE accordingly!")


In [ ]:
# Example workflow: Process one coin image (if available)
coin_image_dir = "Dataset/coinImages"  # Change path if coin images are elsewhere

if os.path.exists(coin_image_dir):
    coin_images = glob.glob(os.path.join(coin_image_dir, "*.jpg"))
    
    if coin_images:
        test_coin_path = coin_images[0]
        print(f"Processing: {os.path.basename(test_coin_path)}")
        
        # Load and undistort
        coin_img = cv2.imread(test_coin_path)
        coin_undistorted = cv2.remap(coin_img, mapx, mapy, cv2.INTER_LINEAR)
        
        # Detect coins
        circles = detect_coins_hough(coin_undistorted, min_radius=15, max_radius=60)
        
        if len(circles) > 0:
            classified, total = classify_coins_by_radius(circles, RADIUS_TO_VALUE)
            print(f"\nDetected {len(circles)} coins")
            for x, y, r, val in classified:
                print(f"  Coin at ({x}, {y}): radius={r}px, value={val}")
            print(f"Total value detected: {total}")
        else:
            print("No coins detected. Adjust parameters in detect_coins_hough().")
    else:
        print(f"No .jpg files found in {coin_image_dir}")
else:
    print(f"Coin images directory not found: {coin_image_dir}")
    print("Capture coin images first or provide the correct path.")


## **Lab Workflow Summary**

**✓ Steps Completed:**
1. **Calibration** — Used existing stereo dataset to compute camera matrix K and distortion coefficients
2. **Verification** — Tested undistortion on sample image; edges should look straight

**→ Next Steps:**
1. Capture **10+ coin images** (or use existing coin dataset if available) with uniform lighting
2. Measure **coin radii in pixels** from undistorted images → map to coin values
3. Edit `RADIUS_TO_VALUE` mapping with your actual measurements
4. Run coin detection on all coin images
5. **Evaluation metric:** Calculate accuracy = (detected_total_value / true_total_value) × 100%

**Expected Outputs:**
- `calibration_results/camera_calibration.npz` — Saved K and dist parameters
- `calibration_results/calibration_params.txt` — Human-readable parameters
- `calibration_results/undistortion_example.png` — Visual verification
- Coin detection results with detection accuracy